# GAN-based Synthetic Data Generation for Class Balancing

This notebook demonstrates the complete workflow:
1. Load encoded data
2. **Normalize** encoded data to [0,1] range and save as CSV
3. Train CTGAN on normalized minority class data
4. Generate synthetic data (normalized) and save as CSV
5. Combine original + synthetic normalized data and save as CSV

## Why Normalize Before GAN Training?
- GANs (neural networks) work best with values in [0,1] range
- Prevents gradient issues during training
- All features contribute equally

## Step 1: Import Libraries and Load Data

In [9]:
import pandas as pd
import numpy as np
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata
import warnings
warnings.filterwarnings('ignore')

# Load the original encoded data 
encoded_df = pd.read_csv(r'../data/Final_Encoded.csv')

print("Original Encoded Data")
print(f"Shape: {encoded_df.shape}")
print(f"\nFirst 5 rows:")
encoded_df.head()

Original Encoded Data
Shape: (1481, 31)

First 5 rows:


,Age,Gender,Where_live,AVG_Sleep_Per_Night,Regular_Bed_time,Exam_Night_Bed_Time,Holiday_Bed_Time,Regular_WakeUp_Time,Holiday_WakeUp_Time,Have_Regular_Bed_Time,...,Class_Attendance,Sleepiness_During_Class,Skip_Class_for_Sleep,Focus_on_Academic_Task,Impact_of_Sleep_on_Academic,Current_CGPA5,CGPA3_Class,Aware_of_Recomamended_Sleep,Use_Sleep_Tracking_Devices,Sacrifices_Sleep_for_Academics
0,2,1,0,3,4,4,4,2,2,0,...,1,2,0,0,3,2,1,1,0,1
1,2,0,1,1,3,1,1,1,2,0,...,1,0,1,2,3,3,1,1,0,1
2,2,0,1,0,4,4,4,1,2,0,...,0,0,1,0,0,3,1,1,0,1
3,3,0,1,2,2,3,4,1,2,0,...,0,0,0,4,1,3,1,1,0,1
4,2,0,0,1,3,3,3,1,1,0,...,3,1,0,2,1,3,1,0,0,1


In [10]:
# Check class distribution
TARGET_COLUMN = 'CGPA3_Class'
MINORITY_CLASS = 2  # <3.0 CGPA

print("Class Distribution (CGPA3_Class):")
print("="*40)
class_counts = encoded_df[TARGET_COLUMN].value_counts().sort_index()
for cls, count in class_counts.items():
    pct = count / len(encoded_df) * 100
    label = {0: '3.5-4.0', 1: '3.0-3.49', 2: '<3.0'}[cls]
    marker = ' <-- MINORITY (needs augmentation)' if cls == MINORITY_CLASS else ''
    print(f"Class {cls} ({label}): {count} samples ({pct:.1f}%){marker}")

print(f"\nTotal samples: {len(encoded_df)}")

Class Distribution (CGPA3_Class):
Class 0 (3.5-4.0): 655 samples (44.2%)
Class 1 (3.0-3.49): 608 samples (41.1%)
Class 2 (<3.0): 218 samples (14.7%) <-- MINORITY (needs augmentation)

Total samples: 1481


## Step 2: Normalize Encoded Data to [0,1] Range

We use Min-Max normalization: `normalized = (value - min) / (max - min)`

Since all our encoded values start from 0, this simplifies to: `normalized = value / max`

In [11]:
# Store max values for each column (needed for denormalization later)
max_values = encoded_df.max()

print("Max values per column (used for normalization):")
print("="*50)
for col in encoded_df.columns:
    print(f"{col}: max = {max_values[col]}")

Max values per column (used for normalization):
Age: max = 3
Gender: max = 1
Where_live: max = 2
AVG_Sleep_Per_Night: max = 3
Regular_Bed_time: max = 4
Exam_Night_Bed_Time: max = 4
Holiday_Bed_Time: max = 4
Regular_WakeUp_Time: max = 2
Holiday_WakeUp_Time: max = 2
Have_Regular_Bed_Time: max = 1
Daytime_Nap: max = 3
Struggle_to_Sleep: max = 4
Sleep_Condition: max = 4
Electronic_Devices_Before_Bed: max = 1
Consume_Caffeine_Night: max = 1
Dinnar_Time: max = 2
Smoke: max = 1
Sleep_Affecting_Drugs: max = 1
Daily_Academics_Time_Spend: max = 4
Main_Reason_for_Insufficient_Sleep: max = 6
Rate_Sleep_Quality: max = 4
Class_Attendance: max = 3
Sleepiness_During_Class: max = 3
Skip_Class_for_Sleep: max = 1
Focus_on_Academic_Task: max = 4
Impact_of_Sleep_on_Academic: max = 4
Current_CGPA5: max = 4
CGPA3_Class: max = 2
Aware_of_Recomamended_Sleep: max = 1
Use_Sleep_Tracking_Devices: max = 1
Sacrifices_Sleep_for_Academics: max = 1


In [12]:
# Normalize all data to [0, 1] range
normalized_df = encoded_df.copy()

for col in normalized_df.columns:
    if max_values[col] > 0:
        normalized_df[col] = normalized_df[col] / max_values[col]
    # If max is 0, values are already 0

print("Normalized Data")
print(f"Shape: {normalized_df.shape}")
print(f"\nValue range: [{normalized_df.min().min():.4f}, {normalized_df.max().max():.4f}]")
print(f"\nFirst 5 rows:")
normalized_df.head()

Normalized Data
Shape: (1481, 31)

Value range: [0.0000, 1.0000]

First 5 rows:


,Age,Gender,Where_live,AVG_Sleep_Per_Night,Regular_Bed_time,Exam_Night_Bed_Time,Holiday_Bed_Time,Regular_WakeUp_Time,Holiday_WakeUp_Time,Have_Regular_Bed_Time,...,Class_Attendance,Sleepiness_During_Class,Skip_Class_for_Sleep,Focus_on_Academic_Task,Impact_of_Sleep_on_Academic,Current_CGPA5,CGPA3_Class,Aware_of_Recomamended_Sleep,Use_Sleep_Tracking_Devices,Sacrifices_Sleep_for_Academics
0,0.666667,1.0,0.0,1.000000,1.00,1.00,1.00,1.0,1.0,0.0,...,0.333333,0.666667,0.0,0.0,0.75,0.50,0.5,1.0,0.0,1.0
1,0.666667,0.0,0.5,0.333333,0.75,0.25,0.25,0.5,1.0,0.0,...,0.333333,0.000000,1.0,0.5,0.75,0.75,0.5,1.0,0.0,1.0
2,0.666667,0.0,0.5,0.000000,1.00,1.00,1.00,0.5,1.0,0.0,...,0.000000,0.000000,1.0,0.0,0.00,0.75,0.5,1.0,0.0,1.0
3,1.000000,0.0,0.5,0.666667,0.50,0.75,1.00,0.5,1.0,0.0,...,0.000000,0.000000,0.0,1.0,0.25,0.75,0.5,1.0,0.0,1.0
4,0.666667,0.0,0.0,0.333333,0.75,0.75,0.75,0.5,0.5,0.0,...,1.000000,0.333333,0.0,0.5,0.25,0.75,0.5,0.0,0.0,1.0


In [13]:
# Save normalized data to CSV
NORMALIZED_PATH = r'../data/Final_Encoded_Normalized.csv'
normalized_df.to_csv(NORMALIZED_PATH, index=False)

print(f"✓ Normalized encoded data saved to:")
print(f"  {NORMALIZED_PATH}")
print(f"\nShape: {normalized_df.shape}")

✓ Normalized encoded data saved to:
  ../data/Final_Encoded_Normalized.csv

Shape: (1481, 31)


## Step 3: Extract Minority Class Data for GAN Training

In [14]:
# Extract minority class (Class 2 = <3.0 CGPA) from NORMALIZED data
minority_normalized = normalized_df[normalized_df[TARGET_COLUMN] == (MINORITY_CLASS / max_values[TARGET_COLUMN])].copy()

print(f"Minority Class Data (Normalized)")
print(f"Shape: {minority_normalized.shape}")
print(f"\nTarget column value (normalized): {MINORITY_CLASS / max_values[TARGET_COLUMN]:.4f}")
print(f"\nFirst 5 rows:")
minority_normalized.head()

Minority Class Data (Normalized)
Shape: (218, 31)

Target column value (normalized): 1.0000

First 5 rows:


,Age,Gender,Where_live,AVG_Sleep_Per_Night,Regular_Bed_time,Exam_Night_Bed_Time,Holiday_Bed_Time,Regular_WakeUp_Time,Holiday_WakeUp_Time,Have_Regular_Bed_Time,...,Class_Attendance,Sleepiness_During_Class,Skip_Class_for_Sleep,Focus_on_Academic_Task,Impact_of_Sleep_on_Academic,Current_CGPA5,CGPA3_Class,Aware_of_Recomamended_Sleep,Use_Sleep_Tracking_Devices,Sacrifices_Sleep_for_Academics
16,0.666667,0.0,0.5,0.000000,1.00,1.0,1.00,0.5,1.0,0.0,...,0.333333,0.666667,1.0,0.5,0.75,1.0,1.0,1.0,0.0,1.0
19,0.666667,0.0,1.0,0.333333,1.00,1.0,1.00,1.0,1.0,0.0,...,0.666667,0.666667,1.0,0.5,0.75,1.0,1.0,1.0,0.0,1.0
22,0.666667,0.0,0.0,0.000000,1.00,1.0,1.00,0.5,1.0,0.0,...,0.333333,0.666667,0.0,0.5,0.75,1.0,1.0,1.0,0.0,1.0
25,0.666667,0.0,0.0,0.333333,1.00,1.0,1.00,0.5,1.0,0.0,...,0.666667,0.333333,0.0,0.5,0.50,1.0,1.0,1.0,1.0,1.0
27,0.666667,1.0,0.0,0.333333,0.75,1.0,0.75,0.5,0.5,1.0,...,0.000000,0.000000,0.0,0.5,0.50,1.0,1.0,1.0,0.0,0.0


## Step 4: Setup and Train CTGAN on Normalized Data

In [15]:
# Create metadata for SDV
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(minority_normalized)

# All columns are numerical after normalization
for col in minority_normalized.columns:
    metadata.update_column(col, sdtype='numerical')

print("Metadata configured for CTGAN")
print(f"Number of columns: {len(minority_normalized.columns)}")
print(f"All columns set as 'numerical' type")

Metadata configured for CTGAN
Number of columns: 31
All columns set as 'numerical' type


In [16]:
# Initialize CTGAN
synthesizer = CTGANSynthesizer(
    metadata,
    epochs=500,
    batch_size=50,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    generator_lr=2e-4,
    discriminator_lr=2e-4,
    verbose=True
)

print("CTGAN Configuration:")
print(f"  Epochs: 500")
print(f"  Batch size: 50")
print(f"  Generator architecture: (256, 256)")
print(f"  Discriminator architecture: (256, 256)")
print(f"  Learning rate: 2e-4")

CTGAN Configuration:
  Epochs: 500
  Batch size: 50
  Generator architecture: (256, 256)
  Discriminator architecture: (256, 256)
  Learning rate: 2e-4


In [17]:
# Train CTGAN on normalized minority class data
print(f"Training CTGAN on {len(minority_normalized)} normalized samples...")
print("This may take 2-3 minutes...\n")

synthesizer.fit(minority_normalized)

print("\n✓ CTGAN training completed!")

Training CTGAN on 218 normalized samples...
This may take 2-3 minutes...



Gen. (-6.24) | Discrim. (0.66): 100%|██████████| 500/500 [02:00<00:00,  4.16it/s] 


✓ CTGAN training completed!


## Step 5: Generate Synthetic Normalized Data

In [18]:
# Calculate how many samples to generate
majority_count = encoded_df[TARGET_COLUMN].value_counts().max()  # 655
minority_count = len(minority_normalized)  # 218
samples_to_generate = majority_count - minority_count  # 437

print(f"Majority class count: {majority_count}")
print(f"Minority class count: {minority_count}")
print(f"Samples to generate: {samples_to_generate}")

Majority class count: 655
Minority class count: 218
Samples to generate: 437


In [19]:
# Generate synthetic samples (these are in NORMALIZED form)
synthetic_normalized = synthesizer.sample(num_rows=samples_to_generate)

print(f"Generated Synthetic Data (NORMALIZED)")
print(f"Shape: {synthetic_normalized.shape}")
print(f"\nValue range: [{synthetic_normalized.min().min():.4f}, {synthetic_normalized.max().max():.4f}]")
print(f"\nFirst 10 rows:")
synthetic_normalized.head(10)

Generated Synthetic Data (NORMALIZED)
Shape: (437, 31)

Value range: [0.0000, 1.0000]

First 10 rows:


,Age,Gender,Where_live,AVG_Sleep_Per_Night,Regular_Bed_time,Exam_Night_Bed_Time,Holiday_Bed_Time,Regular_WakeUp_Time,Holiday_WakeUp_Time,Have_Regular_Bed_Time,...,Class_Attendance,Sleepiness_During_Class,Skip_Class_for_Sleep,Focus_on_Academic_Task,Impact_of_Sleep_on_Academic,Current_CGPA5,CGPA3_Class,Aware_of_Recomamended_Sleep,Use_Sleep_Tracking_Devices,Sacrifices_Sleep_for_Academics
0,0.673384,0.0,0.8,0.762258,1.00,1.00,1.00,0.5,1.0,0.0,...,1.000000,0.725668,1.0,0.04,0.00,1.0,1.0,1.0,0.0,0.0
1,0.352772,0.0,0.9,0.606016,0.83,1.00,1.00,0.5,1.0,0.0,...,0.397595,0.227556,1.0,0.78,0.87,1.0,1.0,1.0,0.0,1.0
2,0.667420,0.0,0.0,0.605379,0.98,0.75,0.96,1.0,1.0,1.0,...,0.560640,0.593929,1.0,0.80,0.35,1.0,1.0,1.0,0.0,1.0
3,0.664518,0.0,0.4,0.629921,1.00,0.97,0.94,0.6,1.0,0.0,...,0.337088,0.056669,0.0,0.58,0.34,1.0,1.0,1.0,0.0,0.0
4,0.367111,0.0,0.0,0.610040,1.00,1.00,0.98,0.6,1.0,0.0,...,0.121211,0.000000,1.0,0.47,0.30,1.0,1.0,1.0,0.0,1.0
5,0.688011,0.0,0.0,0.618023,1.00,0.81,1.00,0.9,1.0,0.0,...,0.422032,0.687941,1.0,0.53,0.52,1.0,1.0,1.0,0.0,0.0
6,0.333333,0.0,0.9,0.593921,1.00,1.00,1.00,0.5,1.0,0.0,...,0.450844,0.000000,1.0,0.47,0.13,1.0,1.0,1.0,0.0,0.0
7,0.673966,1.0,0.9,0.774234,1.00,0.99,1.00,0.5,1.0,0.0,...,0.071153,0.000000,1.0,0.83,0.32,1.0,1.0,1.0,0.0,1.0
8,0.690574,0.0,0.0,0.704327,0.79,0.65,0.98,0.5,1.0,0.0,...,0.055546,0.549324,1.0,0.00,0.26,1.0,1.0,1.0,0.0,1.0
9,0.653761,1.0,0.9,0.658825,1.00,0.98,0.99,0.9,1.0,0.0,...,0.180584,0.720747,0.0,0.55,0.23,1.0,1.0,1.0,0.0,1.0


In [20]:
# Clip values to [0, 1] range and fix the target column
for col in synthetic_normalized.columns:
    synthetic_normalized[col] = synthetic_normalized[col].clip(0, 1)

# Ensure target column is exactly the minority class value (normalized)
synthetic_normalized[TARGET_COLUMN] = MINORITY_CLASS / max_values[TARGET_COLUMN]

print("Synthetic data clipped to [0, 1] range")
print(f"Target column (CGPA3_Class) set to: {MINORITY_CLASS / max_values[TARGET_COLUMN]:.4f}")
print(f"\nValue range after clipping: [{synthetic_normalized.min().min():.4f}, {synthetic_normalized.max().max():.4f}]")

Synthetic data clipped to [0, 1] range
Target column (CGPA3_Class) set to: 1.0000

Value range after clipping: [0.0000, 1.0000]


In [22]:
# Save synthetic NORMALIZED data to CSV
SYNTHETIC_NORMALIZED_PATH = r'../data/Synthetic_Minority_Normalized.csv'
synthetic_normalized.to_csv(SYNTHETIC_NORMALIZED_PATH, index=False)

print(f"✓ Synthetic NORMALIZED data saved to:")
print(f"  {SYNTHETIC_NORMALIZED_PATH}")
print(f"\nShape: {synthetic_normalized.shape}")
print(f"\nFirst 5 rows:")
synthetic_normalized.head()

✓ Synthetic NORMALIZED data saved to:
  ../data/Synthetic_Minority_Normalized.csv

Shape: (437, 31)

First 5 rows:


,Age,Gender,Where_live,AVG_Sleep_Per_Night,Regular_Bed_time,Exam_Night_Bed_Time,Holiday_Bed_Time,Regular_WakeUp_Time,Holiday_WakeUp_Time,Have_Regular_Bed_Time,...,Class_Attendance,Sleepiness_During_Class,Skip_Class_for_Sleep,Focus_on_Academic_Task,Impact_of_Sleep_on_Academic,Current_CGPA5,CGPA3_Class,Aware_of_Recomamended_Sleep,Use_Sleep_Tracking_Devices,Sacrifices_Sleep_for_Academics
0,0.673384,0.0,0.8,0.762258,1.00,1.00,1.00,0.5,1.0,0.0,...,1.000000,0.725668,1.0,0.04,0.00,1.0,1.0,1.0,0.0,0.0
1,0.352772,0.0,0.9,0.606016,0.83,1.00,1.00,0.5,1.0,0.0,...,0.397595,0.227556,1.0,0.78,0.87,1.0,1.0,1.0,0.0,1.0
2,0.667420,0.0,0.0,0.605379,0.98,0.75,0.96,1.0,1.0,1.0,...,0.560640,0.593929,1.0,0.80,0.35,1.0,1.0,1.0,0.0,1.0
3,0.664518,0.0,0.4,0.629921,1.00,0.97,0.94,0.6,1.0,0.0,...,0.337088,0.056669,0.0,0.58,0.34,1.0,1.0,1.0,0.0,0.0
4,0.367111,0.0,0.0,0.610040,1.00,1.00,0.98,0.6,1.0,0.0,...,0.121211,0.000000,1.0,0.47,0.30,1.0,1.0,1.0,0.0,1.0


## Step 6: Combine Original + Synthetic Normalized Data

In [23]:
# Combine original normalized data with synthetic normalized data
combined_normalized = pd.concat([normalized_df, synthetic_normalized], ignore_index=True)

print(f"Combined Normalized Dataset")
print(f"Original normalized: {len(normalized_df)} samples")
print(f"Synthetic normalized: {len(synthetic_normalized)} samples")
print(f"Combined total: {len(combined_normalized)} samples")

Combined Normalized Dataset
Original normalized: 1481 samples
Synthetic normalized: 437 samples
Combined total: 1918 samples


In [24]:
# Check new class distribution
print("Class Distribution in Combined Normalized Data:")
print("="*50)

# Get unique normalized values for target
for cls in [0, 1, 2]:
    normalized_cls_value = cls / max_values[TARGET_COLUMN]
    count = len(combined_normalized[combined_normalized[TARGET_COLUMN] == normalized_cls_value])
    pct = count / len(combined_normalized) * 100
    label = {0: '3.5-4.0', 1: '3.0-3.49', 2: '<3.0'}[cls]
    print(f"Class {cls} ({label}): {count} samples ({pct:.1f}%)")

print(f"\nTotal: {len(combined_normalized)} samples")

Class Distribution in Combined Normalized Data:
Class 0 (3.5-4.0): 655 samples (34.2%)
Class 1 (3.0-3.49): 608 samples (31.7%)
Class 2 (<3.0): 655 samples (34.2%)

Total: 1918 samples


In [26]:
# Save combined normalized data to CSV
COMBINED_NORMALIZED_PATH = r'../data/Final_Combined_Normalized.csv'
combined_normalized.to_csv(COMBINED_NORMALIZED_PATH, index=False)

print(f"✓ Combined NORMALIZED data saved to:")
print(f"  {COMBINED_NORMALIZED_PATH}")
print(f"\nShape: {combined_normalized.shape}")

✓ Combined NORMALIZED data saved to:
  ../data/Final_Combined_Normalized.csv

Shape: (1918, 31)


## Step 7: Summary of All Generated Files

In [28]:
print("="*70)
print("SUMMARY OF GENERATED FILES")
print("="*70)

print("\n1. NORMALIZED ENCODED DATA (Full Dataset)")
print(f"   File: Final_Encoded_Normalized.csv")
print(f"   Shape: {normalized_df.shape}")
print(f"   Description: Original encoded data normalized to [0,1] range")

print("\n2. SYNTHETIC MINORITY CLASS (Normalized)")
print(f"   File: Synthetic_Minority_Normalized.csv")
print(f"   Shape: {synthetic_normalized.shape}")
print(f"   Description: GAN-generated synthetic samples for minority class")

print("\n3. COMBINED NORMALIZED DATA (Balanced)")
print(f"   File: Final_Combined_Normalized.csv")
print(f"   Shape: {combined_normalized.shape}")
print(f"   Description: Original + Synthetic data combined (balanced classes)")

print("\n" + "="*70)
print("CLASS BALANCE COMPARISON")
print("="*70)

print("\nBEFORE (Original):")
for cls in [0, 1, 2]:
    count = len(encoded_df[encoded_df[TARGET_COLUMN] == cls])
    pct = count / len(encoded_df) * 100
    print(f"  Class {cls}: {count} ({pct:.1f}%)")

print("\nAFTER (Combined with Synthetic):")
for cls in [0, 1, 2]:
    normalized_cls_value = cls / max_values[TARGET_COLUMN]
    count = len(combined_normalized[combined_normalized[TARGET_COLUMN] == normalized_cls_value])
    pct = count / len(combined_normalized) * 100
    print(f"  Class {cls}: {count} ({pct:.1f}%)")

SUMMARY OF GENERATED FILES

1. NORMALIZED ENCODED DATA (Full Dataset)
   File: Final_Encoded_Normalized.csv
   Shape: (1481, 31)
   Description: Original encoded data normalized to [0,1] range

2. SYNTHETIC MINORITY CLASS (Normalized)
   File: Synthetic_Minority_Normalized.csv
   Shape: (437, 31)
   Description: GAN-generated synthetic samples for minority class

3. COMBINED NORMALIZED DATA (Balanced)
   File: Final_Combined_Normalized.csv
   Shape: (1918, 31)
   Description: Original + Synthetic data combined (balanced classes)

CLASS BALANCE COMPARISON

BEFORE (Original):
  Class 0: 655 (44.2%)
  Class 1: 608 (41.1%)
  Class 2: 218 (14.7%)

AFTER (Combined with Synthetic):
  Class 0: 655 (34.2%)
  Class 1: 608 (31.7%)
  Class 2: 655 (34.2%)


## Step 8: Verify the Normalized Data Files

In [29]:
# Load and verify saved files
print("Verifying saved files...\n")

# Check normalized encoded
verify_normalized = pd.read_csv(NORMALIZED_PATH)
print(f"1. Final_Encoded_Normalized.csv")
print(f"   Shape: {verify_normalized.shape}")
print(f"   Value range: [{verify_normalized.min().min():.4f}, {verify_normalized.max().max():.4f}]")
print(f"   Sample values (Age column): {verify_normalized['Age'].head(5).tolist()}")

print()

# Check synthetic normalized
verify_synthetic = pd.read_csv(SYNTHETIC_NORMALIZED_PATH)
print(f"2. Synthetic_Minority_Normalized.csv")
print(f"   Shape: {verify_synthetic.shape}")
print(f"   Value range: [{verify_synthetic.min().min():.4f}, {verify_synthetic.max().max():.4f}]")
print(f"   Sample values (Age column): {verify_synthetic['Age'].head(5).tolist()}")

print()

# Check combined normalized
verify_combined = pd.read_csv(COMBINED_NORMALIZED_PATH)
print(f"3. Final_Combined_Normalized.csv")
print(f"   Shape: {verify_combined.shape}")
print(f"   Value range: [{verify_combined.min().min():.4f}, {verify_combined.max().max():.4f}]")

Verifying saved files...

1. Final_Encoded_Normalized.csv
   Shape: (1481, 31)
   Value range: [0.0000, 1.0000]
   Sample values (Age column): [0.6666666666666666, 0.6666666666666666, 0.6666666666666666, 1.0, 0.6666666666666666]

2. Synthetic_Minority_Normalized.csv
   Shape: (437, 31)
   Value range: [0.0000, 1.0000]
   Sample values (Age column): [0.6733840656186324, 0.3527724966986544, 0.6674195853177508, 0.6645181681960296, 0.3671111672374101]

3. Final_Combined_Normalized.csv
   Shape: (1918, 31)
   Value range: [0.0000, 1.0000]


In [30]:
# Show sample of normalized vs original encoded data
print("COMPARISON: Original Encoded vs Normalized")
print("="*60)

sample_cols = ['Age', 'Gender', 'AVG_Sleep_Per_Night', 'CGPA3_Class']

print("\nOriginal Encoded (first 5 rows):")
print(encoded_df[sample_cols].head())

print("\nNormalized (first 5 rows):")
print(normalized_df[sample_cols].head())

print("\nSynthetic Normalized (first 5 rows):")
print(synthetic_normalized[sample_cols].head())

COMPARISON: Original Encoded vs Normalized

Original Encoded (first 5 rows):
   Age  Gender  AVG_Sleep_Per_Night  CGPA3_Class
0    2       1                    3            1
1    2       0                    1            1
2    2       0                    0            1
3    3       0                    2            1
4    2       0                    1            1

Normalized (first 5 rows):
        Age  Gender  AVG_Sleep_Per_Night  CGPA3_Class
0  0.666667     1.0             1.000000          0.5
1  0.666667     0.0             0.333333          0.5
2  0.666667     0.0             0.000000          0.5
3  1.000000     0.0             0.666667          0.5
4  0.666667     0.0             0.333333          0.5

Synthetic Normalized (first 5 rows):
        Age  Gender  AVG_Sleep_Per_Night  CGPA3_Class
0  0.673384     0.0             0.762258          1.0
1  0.352772     0.0             0.606016          1.0
2  0.667420     0.0             0.605379          1.0
3  0.664518     0.0    

## Notes

### How to Denormalize (if needed later):
To convert normalized values back to encoded integers:
```python
for col in normalized_df.columns:
    encoded_value = round(normalized_value * max_values[col])
```

### Max Values Reference:
- Age: 3
- Gender: 1
- Where_live: 2
- AVG_Sleep_Per_Night: 3
- CGPA3_Class: 2
- (see encoding.txt for full reference)